In [1]:
from pathlib import Path
for p in Path("data").iterdir():
    print(p)

data\2026q3_form345
data\2026q3_form345.zip


In [2]:
import pandas as pd

FOLDER = Path("data") / "2026q3_form345"

sub = pd.read_csv(FOLDER / "SUBMISSION.tsv", sep="\t", dtype=str)
own = pd.read_csv(FOLDER / "REPORTINGOWNER.tsv", sep="\t", dtype=str)
trans = pd.read_csv(FOLDER / "NONDERIV_TRANS.tsv", sep="\t", dtype=str)

print(sub.shape, own.shape, trans.shape)
trans.head()

(38152, 14) (41346, 13) (59885, 28)


,ACCESSION_NUMBER,NONDERIV_TRANS_SK,SECURITY_TITLE,SECURITY_TITLE_FN,TRANS_DATE,TRANS_DATE_FN,DEEMED_EXECUTION_DATE,DEEMED_EXECUTION_DATE_FN,TRANS_FORM_TYPE,TRANS_CODE,...,TRANS_ACQUIRED_DISP_CD,TRANS_ACQUIRED_DISP_CD_FN,SHRS_OWND_FOLWNG_TRANS,SHRS_OWND_FOLWNG_TRANS_FN,VALU_OWND_FOLWNG_TRANS,VALU_OWND_FOLWNG_TRANS_FN,DIRECT_INDIRECT_OWNERSHIP,DIRECT_INDIRECT_OWNERSHIP_FN,NATURE_OF_OWNERSHIP,NATURE_OF_OWNERSHIP_FN
0,0001213900-26-105429,9078292,Common Stock,NaN,28-SEP-2026,NaN,NaN,NaN,4,S,...,D,NaN,4384095.0,F3,NaN,NaN,D,NaN,NaN,NaN
1,0001829126-26-010633,9078001,Common Stock,NaN,28-SEP-2026,NaN,NaN,NaN,4,J,...,D,NaN,460000.0,NaN,NaN,NaN,D,NaN,NaN,NaN
2,0001829126-26-010633,9078000,Common Stock,NaN,28-SEP-2026,NaN,NaN,NaN,4,J,...,D,NaN,465000.0,NaN,NaN,NaN,D,NaN,NaN,NaN
3,0001829126-26-010632,9078620,Common Stock,NaN,28-SEP-2026,NaN,NaN,NaN,4,J,...,D,NaN,398000.0,NaN,NaN,NaN,D,NaN,NaN,NaN
4,0002092936-26-000006,9078619,Ordinary Participation Certificates (CEMEX.CPO),NaN,30-SEP-2026,NaN,NaN,NaN,4,S,...,D,NaN,584930.0,F3,NaN,NaN,D,NaN,NaN,NaN


In [3]:
print(trans["TRANS_CODE"].value_counts(dropna=False))

buys = trans[trans["TRANS_CODE"] == "P"]
print("open-market purchases:", len(buys), "of", len(trans))

print(own["RPTOWNER_RELATIONSHIP"].value_counts().head(10))

TRANS_CODE
S      25441
A       9817
M       7433
F       6802
P       5184
J       1679
D       1382
G       1035
C        903
L         65
X         56
U         38
I         23
W         14
Z          8
NaN        4
O          1
Name: count, dtype: int64
open-market purchases: 5184 of 59885
RPTOWNER_RELATIONSHIP
Officer                             15444
Director                            14384
Director,Officer                     4211
TenPercentOwner                      3894
Director,TenPercentOwner             1160
Director,Officer,TenPercentOwner      897
Other                                 874
Director,Other                        149
TenPercentOwner,Other                 131
Officer,TenPercentOwner                58
Name: count, dtype: int64


In [4]:
merged = buys.merge(own, on="ACCESSION_NUMBER", how="left")
print("purchase rows before join:", len(buys))
print("purchase rows after join: ", len(merged))
print(list(sub.columns))

purchase rows before join: 5184
purchase rows after join:  7043
['ACCESSION_NUMBER', 'FILING_DATE', 'PERIOD_OF_REPORT', 'DATE_OF_ORIG_SUB', 'NO_SECURITIES_OWNED', 'NOT_SUBJECT_SEC16', 'FORM3_HOLDINGS_REPORTED', 'FORM4_TRANS_REPORTED', 'DOCUMENT_TYPE', 'ISSUERCIK', 'ISSUERNAME', 'ISSUERTRADINGSYMBOL', 'REMARKS', 'AFF10B5ONE']


In [5]:
cols = ["ACCESSION_NUMBER", "FILING_DATE", "DOCUMENT_TYPE",
        "ISSUERCIK", "ISSUERNAME", "ISSUERTRADINGSYMBOL"]
full = merged.merge(sub[cols], on="ACCESSION_NUMBER", how="left")

print("rows:", len(full))
print(full["DOCUMENT_TYPE"].value_counts(dropna=False))
full[["ISSUERTRADINGSYMBOL", "RPTOWNERNAME", "RPTOWNER_RELATIONSHIP",
      "TRANS_DATE", "FILING_DATE", "TRANS_SHARES", "TRANS_PRICEPERSHARE"]].head(10)

rows: 7043
DOCUMENT_TYPE
4      6811
4/A     230
5         2
Name: count, dtype: int64


,ISSUERTRADINGSYMBOL,RPTOWNERNAME,RPTOWNER_RELATIONSHIP,TRANS_DATE,FILING_DATE,TRANS_SHARES,TRANS_PRICEPERSHARE
0,GPI,"Conifer Management, L.L.C.",TenPercentOwner,30-SEP-2026,30-SEP-2026,16514.0,243.42
1,GPI,"Conifer Management, L.L.C.",TenPercentOwner,30-SEP-2026,30-SEP-2026,16343.0,244.61
2,GPI,"Conifer Management, L.L.C.",TenPercentOwner,30-SEP-2026,30-SEP-2026,1040.0,245.1
3,GPI,"Conifer Management, L.L.C.",TenPercentOwner,30-SEP-2026,30-SEP-2026,5352.0,242.65
4,GPI,"Conifer Management, L.L.C.",TenPercentOwner,30-SEP-2026,30-SEP-2026,3151.0,241.04
5,GPI,"Conifer Management, L.L.C.",TenPercentOwner,30-SEP-2026,30-SEP-2026,12342.0,240.31
6,GPI,"Conifer Management, L.L.C.",TenPercentOwner,30-SEP-2026,30-SEP-2026,14717.0,239.37
7,GPI,"Conifer Management, L.L.C.",TenPercentOwner,29-SEP-2026,30-SEP-2026,160.0,244.83
8,GPI,"Conifer Management, L.L.C.",TenPercentOwner,30-SEP-2026,30-SEP-2026,2133.0,238.37
9,GPI,"Conifer Management, L.L.C.",TenPercentOwner,29-SEP-2026,30-SEP-2026,320.0,243.86


In [6]:
d = full.copy()
d["trans_date"] = pd.to_datetime(d["TRANS_DATE"], format="%d-%b-%Y")
d["filing_date"] = pd.to_datetime(d["FILING_DATE"], format="%d-%b-%Y")
d["shares"] = pd.to_numeric(d["TRANS_SHARES"])
d["price"] = pd.to_numeric(d["TRANS_PRICEPERSHARE"])
d["value"] = d["shares"] * d["price"]

d["is_exec"] = d["RPTOWNER_RELATIONSHIP"].str.contains("Officer|Director", na=False)

print(d[["shares", "price", "value"]].isna().sum())
print(d["is_exec"].value_counts())
print(d.groupby("is_exec")["value"].sum())


shares     0
price     50
value     50
dtype: int64
is_exec
True     4319
False    2724
Name: count, dtype: int64
is_exec
False    1.600010e+15
True     2.944407e+10
Name: value, dtype: float64


In [7]:
show = ["ISSUERTRADINGSYMBOL", "RPTOWNERNAME", "RPTOWNER_RELATIONSHIP",
        "TRANS_DATE", "shares", "price", "value"]

print("Biggest 10 rows by value:")
print(d.sort_values("value", ascending=False)[show].head(10))

print("\nRows with no price (first 5):")
print(d[d["price"].isna()][show].head(5))

Biggest 10 rows by value:
     ISSUERTRADINGSYMBOL                           RPTOWNERNAME  \
6430                FINS     MetLife Investment Management, LLC   
4990                REBN                           Lim Jung Jae   
6739                SVRE              VisionWave Holdings, Inc.   
5551                 NMM                       Frangou Angeliki   
3638                 JEF  SUMITOMO MITSUI FINANCIAL GROUP, INC.   
6073                 JEF  SUMITOMO MITSUI FINANCIAL GROUP, INC.   
3112                MAIR                           KC Armada LP   
3111                MAIR                     BERTARELLI ERNESTO   
3514                SCTH                          SITRA J SCOTT   
6920                 NaN              ANGELO GORDON & CO., L.P.   

                 RPTOWNER_RELATIONSHIP   TRANS_DATE        shares  \
6430                   TenPercentOwner  08-JUL-2026  4.000000e+07   
4990                  Director,Officer  14-JUL-2026  1.313870e+05   
6739                         

In [8]:
key = ["ACCESSION_NUMBER", "NONDERIV_TRANS_SK"]
trades = d.drop_duplicates(key)
print("one row per transaction:", len(trades), "(should be 5184)")

print("Total value, one row per transaction:", trades["value"].sum())

print("\nPurchases with price above $1,000:")
hi = trades[trades["price"] > 1000]
print(len(hi))
print(hi[["ISSUERTRADINGSYMBOL", "RPTOWNERNAME", "shares", "price"]].sort_values("price", ascending=False))

one row per transaction: 5184 (should be 5184)
Total value, one row per transaction: 1600032411271172.8

Purchases with price above $1,000:
12
     ISSUERTRADINGSYMBOL                        RPTOWNERNAME      shares  \
6430                FINS  MetLife Investment Management, LLC  40000000.0   
5551                 NMM                    Frangou Angeliki      1131.0   
4990                REBN                        Lim Jung Jae    131387.0   
3514                SCTH                       SITRA J SCOTT      1000.0   
2560                NONE                      HARVEY ELLEN D         0.0   
2559                NONE                      HARVEY ELLEN D         0.0   
2558                NONE                      HARVEY ELLEN D         0.0   
4704                 NVR                       Oliver George        63.0   
5187                APCX                       LORD ALBERT L     20000.0   
5186                APCX                       LORD ALBERT L     20000.0   
3780               FC

In [9]:
import yfinance as yf

checks = hi[hi["ISSUERTRADINGSYMBOL"] != "NONE"][
    ["ISSUERTRADINGSYMBOL", "trans_date", "price"]].drop_duplicates()

rows = []
for _, r in checks.iterrows():
    t = r["ISSUERTRADINGSYMBOL"]
    start = r["trans_date"] - pd.Timedelta(days=3)
    end = r["trans_date"] + pd.Timedelta(days=4)
    h = yf.Ticker(t).history(start=start, end=end)
    mkt = h["Close"].mean() if len(h) else None
    rows.append((t, r["trans_date"].date(), r["price"], mkt))

pd.DataFrame(rows, columns=["ticker", "trade_date", "reported_price", "market_close_avg"])

,ticker,trade_date,reported_price,market_close_avg
0,SCTH,2026-08-21,100000.00,21.857000
1,FCNCA,2026-08-19,2167.80,2190.169678
2,FCNCA,2026-08-19,2168.13,2190.169678
3,NVR,2026-08-07,6431.51,6324.071973
4,REBN,2026-07-14,180000.00,1.391000
5,APCX,2026-07-31,2895.00,0.277400
6,APCX,2026-08-03,2900.00,0.343800
7,NMM,2026-07-23,748119.00,75.164194
8,FINS,2026-07-08,40000000.00,12.291390


In [10]:
print("unique tickers:", trades["ISSUERTRADINGSYMBOL"].nunique())
print(trades["ISSUERTRADINGSYMBOL"].isna().sum(), "rows with no ticker")

unique tickers: 1049
52 rows with no ticker


In [11]:
import time

tickers = sorted(t for t in trades["ISSUERTRADINGSYMBOL"].dropna().unique() if t != "NONE")
start = trades["trans_date"].min() - pd.Timedelta(days=5)
end = trades["trans_date"].max() + pd.Timedelta(days=5)
print(len(tickers), "tickers;", start.date(), "to", end.date())

1048 tickers; 2006-08-23 to 2026-10-05


In [12]:
old = trades[trades["trans_date"] < "2026-06-01"]
print("rows with trade date before June 2026:", len(old))
print(old[["ISSUERTRADINGSYMBOL", "RPTOWNERNAME", "TRANS_DATE", "FILING_DATE",
           "DOCUMENT_TYPE", "shares", "price"]].sort_values("TRANS_DATE").head(15))

lag = (trades["filing_date"] - trades["trans_date"]).dt.days
print("\nDays between trade and filing:")
print(lag.describe())

rows with trade date before June 2026: 157
     ISSUERTRADINGSYMBOL                               RPTOWNERNAME  \
2669               PBLSX  BlueArc Core Alternatives Management, LLC   
6280              RANGER                                Glass Bryan   
3327                 NaN          Manulife Private Credit Plus Fund   
4139                 CFG                          Swift Christopher   
4019             AB-Lend                            PM Alpha II DAC   
1824                 CZR                             TOMICK DAVID P   
6261              RANGER                                Glass Bryan   
5401                FDBC                           DelVecchio Rocco   
6246              RANGER                                Glass Bryan   
6273              RANGER                                Glass Bryan   
6281              RANGER                                Glass Bryan   
4138                 CFG                          Swift Christopher   
6240              RANGER          

In [13]:
show = ["ISSUERTRADINGSYMBOL", "RPTOWNERNAME", "trans_date", "filing_date", "shares", "price"]
print(trades.sort_values("trans_date")[show].head(8))

lag = (trades["filing_date"] - trades["trans_date"]).dt.days
for n in [5, 10, 30, 90]:
    print("lag over", n, "days:", (lag > n).sum(), "of", len(lag))

     ISSUERTRADINGSYMBOL                  RPTOWNERNAME trans_date filing_date  \
2915                SHMD  Schuetz Arthur Josef Hermann 2006-08-28  2026-08-31   
4475                CAKE                   Ames Edie A 2018-03-12  2026-08-12   
2565                SMID                SMITH ASHLEY B 2018-12-07  2026-09-02   
2421                SMID                SMITH ASHLEY B 2018-12-07  2026-09-04   
2641                 SYK               BRAINERD MARY K 2019-06-26  2026-09-02   
2640                 SYK               BRAINERD MARY K 2019-09-12  2026-09-02   
2639                 SYK               BRAINERD MARY K 2020-05-12  2026-09-02   
2638                 SYK               BRAINERD MARY K 2020-06-12  2026-09-02   

       shares   price  
2915  15200.0    3.23  
4475    500.0   49.51  
2565   1275.0    7.69  
2421   1275.0    7.69  
2641     63.0  202.21  
2640     11.0  221.03  
2639     36.0  193.21  
2638      3.0  185.65  
lag over 5 days: 590 of 5184
lag over 10 days: 351 of 

In [14]:
step = trades.copy()
log = [("purchases (code P, one row per trade)", len(step))]

step = step[step["ISSUERTRADINGSYMBOL"].notna() & (step["ISSUERTRADINGSYMBOL"] != "NONE")]
log.append(("has a real ticker", len(step)))

step = step[step["price"].notna() & (step["shares"] > 0)]
log.append(("has price and shares above 0", len(step)))

lagd = (step["filing_date"] - step["trans_date"]).dt.days
step = step[lagd <= 10]
log.append(("filed within 10 days of the trade", len(step)))

step = step[step["DOCUMENT_TYPE"] == "4"]
log.append(("original Form 4 only (no amendments)", len(step)))

clean = step.copy()
print(pd.DataFrame(log, columns=["step", "rows left"]))
print(clean["trans_date"].min().date(), "to", clean["trans_date"].max().date())
print(clean["ISSUERTRADINGSYMBOL"].nunique(), "tickers")

                                    step  rows left
0  purchases (code P, one row per trade)       5184
1                      has a real ticker       4968
2           has price and shares above 0       4926
3      filed within 10 days of the trade       4627
4   original Form 4 only (no amendments)       4547
2026-06-22 to 2026-09-30
971 tickers


In [15]:
import numpy as np

tickers = sorted(clean["ISSUERTRADINGSYMBOL"].unique())
yahoo = {t: t.replace(".", "-") for t in tickers}   # Yahoo writes BRK.B as BRK-B
start = clean["trans_date"].min() - pd.Timedelta(days=5)
end = clean["trans_date"].max() + pd.Timedelta(days=5)

frames = []
names = list(yahoo.values())
for i in range(0, len(names), 100):
    batch = names[i:i+100]
    try:
        df = yf.download(batch, start=start, end=end, auto_adjust=False,
                         progress=False)["Close"]
        frames.append(df)
    except Exception as e:
        print("batch failed at", i, e)
    time.sleep(2)

prices = pd.concat(frames, axis=1)
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices.to_csv("data/prices_2026q3.csv")
print(prices.shape)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AXIA3"}}}
$AXIA3: possibly delisted; no timezone found
$ADIL: possibly delisted; no price data found  (1d 2026-06-17 00:00:00 -> 2026-10-05 00:00:00)
$ACLOX: possibly delisted; no price data found  (1d 2026-06-17 00:00:00 -> 2026-10-05 00:00:00)

3 Failed downloads:
['AXIA3']: possibly delisted; no timezone found
['ADIL', 'ACLOX']: possibly delisted; no price data found  (1d 2026-06-17 00:00:00 -> 2026-10-05 00:00:00)
$CDR: possibly delisted; no timezone found

1 Failed download:
['CDR']: possibly delisted; no timezone found
$CYBN: possibly delisted; no price data found  (1d 2026-06-17 00:00:00 -> 2026-10-05 00:00:00)
$CTKYY: possibly delisted; no timezone found

2 Failed downloads:
['CYBN']: possibly delisted; no price data found  (1d 2026-06-17 00:00:00 -> 2026-10-05 00:00:00)
['CTKYY']: possibly delisted; no timezone found
$EDAP: possibly delisted; no price data foun

(75, 971)


In [16]:
px = prices.sort_index()

def mkt_close(t, date):
    col = yahoo[t]
    if col not in px.columns:
        return np.nan
    s = px[col].dropna()
    s = s[s.index <= date]
    return s.iloc[-1] if len(s) else np.nan

clean = clean.copy()
clean["mkt"] = [mkt_close(t, d) for t, d in zip(clean["ISSUERTRADINGSYMBOL"], clean["trans_date"])]
clean["ratio"] = clean["price"] / clean["mkt"]

print("no market price found:", clean["mkt"].isna().sum())
bad = clean[(clean["ratio"] > 2) | (clean["ratio"] < 0.5)]
print("reported price off by more than 2x:", len(bad))
bad[["ISSUERTRADINGSYMBOL", "RPTOWNERNAME", "trans_date", "price", "mkt", "ratio"]].head(15)

no market price found: 178


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.item(), a.any() or a.all().

In [17]:
px = prices.sort_index()
px = px.loc[:, ~px.columns.duplicated()]   # keep one copy of any repeated ticker
print("repeated columns removed:", prices.shape[1] - px.shape[1])

def mkt_close(t, date):
    col = yahoo[t]
    if col not in px.columns:
        return np.nan
    s = px[col].dropna()
    s = s[s.index <= date]
    return float(s.iloc[-1]) if len(s) else np.nan

clean = clean.copy()
clean["mkt"] = [mkt_close(t, d) for t, d in zip(clean["ISSUERTRADINGSYMBOL"], clean["trans_date"])]
clean["ratio"] = clean["price"] / clean["mkt"]

print("no market price found:", clean["mkt"].isna().sum())
bad = clean[(clean["ratio"] > 2) | (clean["ratio"] < 0.5)]
print("reported price off by more than 2x:", len(bad))
bad[["ISSUERTRADINGSYMBOL", "RPTOWNERNAME", "trans_date", "price", "mkt", "ratio"]].head(15)

repeated columns removed: 1
no market price found: 178
reported price off by more than 2x: 326


,ISSUERTRADINGSYMBOL,RPTOWNERNAME,trans_date,price,mkt,ratio
100,BBD,Di Marcello Francesco,2026-09-29,17.55,3.360000,5.223214
101,BBD,Di Marcello Francesco,2026-09-29,17.56,3.360000,5.226191
102,BBD,Di Marcello Francesco,2026-09-28,17.66,3.350000,5.271642
103,BBD,Di Marcello Francesco,2026-09-28,17.65,3.350000,5.268657
655,MNSO,Zhang Jingjing Eason,2026-09-29,2.26,9.220000,0.245119
727,LILA,MALONE JOHN C,2026-09-28,20.43,8.490000,2.406360
728,LILA,MALONE JOHN C,2026-09-25,20.44,8.480000,2.410377
729,LILA,MALONE JOHN C,2026-09-24,20.45,8.600000,2.377907
925,NCT,Zhu Muchun,2026-09-25,0.40,4.430000,0.090293
926,MSC,HO LAWRENCE YAU LUNG,2026-09-23,0.20,0.800000,0.250000


In [18]:
ok = clean[clean["mkt"].notna()]
r = ok["ratio"]
close10 = ((r > 0.9) & (r < 1.1)).sum()
middle = (((r >= 0.5) & (r <= 0.9)) | ((r >= 1.1) & (r <= 2))).sum()
print("within 10% of market:", close10, "of", len(r))
print("between 10% and 2x off:", middle)
print()
print(bad["ISSUERTRADINGSYMBOL"].value_counts().head(12))

within 10% of market: 3703 of 4369
between 10% and 2x off: 340

ISSUERTRADINGSYMBOL
TSM     109
BBD      27
INM      16
FGBI     15
LILA     11
ENHA     11
LFT       8
PFSA      8
GAM       7
WHLR      7
FGL       7
LGHL      6
Name: count, dtype: int64


In [19]:
final = clean[clean["mkt"].notna() & (clean["ratio"] >= 0.5) & (clean["ratio"] <= 2)].copy()
final["price_check"] = np.where((final["ratio"] > 0.9) & (final["ratio"] < 1.1), "close", "loose")

print(len(final), "transactions kept;", round(len(final) / 5184 * 100, 1), "% of raw purchases")
print(final["price_check"].value_counts())
print(final["is_exec"].value_counts())

key = ["ACCESSION_NUMBER", "NONDERIV_TRANS_SK"]
people = d.merge(final[key + ["price_check"]], on=key, how="inner")
print("owner-level rows:", len(people))

final.to_csv("data/purchases_clean_2026q3.csv", index=False)
people.to_csv("data/purchases_people_2026q3.csv", index=False)

4043 transactions kept; 78.0 % of raw purchases
price_check
close    3703
loose     340
Name: count, dtype: int64
is_exec
True     2873
False    1170
Name: count, dtype: int64
owner-level rows: 5561
